# 09 · Performance Tuning

> **Module 2 · Spark Fundamentals · Lecture 2.7**

In this notebook you will:

- Read physical plans with `explain(True)` and count shuffles
- Tune shuffle partitions, broadcast-join thresholds, and detect and salt skew
- Enable and configure AQE (`spark.sql.adaptive.enabled`) and coalesce shuffle partitions
- Benchmark a native expression against a plain `udf` and a `pandas_udf` with a real measurement
- Tune memory, storage levels, and `cache()` before fan-out — and avoid the common anti-patterns

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


## 1 · Read the plan first

`explain(True)` (identical to `explain(mode="formatted")`) is free — it builds the optimized
physical plan without running the job.

| Plan element | What it tells you |
|---|---|
| `Exchange` | A full network shuffle. Count them. |
| `PushedFilters` under `FileScan` | Whether the filter reached the scan |
| `ReadSchema` under `FileScan` | Whether column pruning worked |
| `BroadcastHashJoin` | A broadcast join was chosen |
| `SortMergeJoin` | Both sides were range-partitioned and sorted |
| `PythonUDF` | A barrier Catalyst cannot optimize past |

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


In [ ]:
# ------------------------------------------------------------------
# A reasonably sized dataset for the benchmark cells.
# ------------------------------------------------------------------
import time

from pyspark.sql import functions as F

N = 1_000_000

bench_df = (
    spark.range(N)
    .withColumn("customer_id", (F.col("id") % 50_000).cast("int"))
    .withColumn("region", F.when(F.col("id") % 3 == 0, F.lit("EMEA"))
                         .when(F.col("id") % 3 == 1, F.lit("APAC"))
                         .otherwise(F.lit("AMER")))
    .withColumn("amount", (F.col("id") % 5000) / 10.0)
)
bench_df.cache().count()
print("rows:", bench_df.count())
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
# A filter + join + aggregation: count the Exchange nodes.
agg = (
    bench_df
    .filter(F.col("amount") > 100)
    .groupBy("region")
    .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders"))
)
agg.explain(True)

## 2 · Shuffle partition tuning

Every shuffle produces `spark.sql.shuffle.partitions` output partitions (default **200**).
Ten rows still create 200 tasks.

```
Recommended partitions ≈ (total shuffle bytes) / (target bytes per task)
target bytes per task ≈ 128–256 MB
```

**Note:** Keep the count near 1–3× the number of executor cores, and never below roughly one
partition per core.

In [ ]:
def timeit(label, fn, repeat=3):
    """Run an action `repeat` times, discard the first (warm-up), report the median."""
    timings = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        timings.append(time.perf_counter() - t0)
    timings.sort()
    median = timings[len(timings) // 2]
    print(f"{label:44s} median={median:7.3f}s  all={[round(t, 3) for t in timings]}")
    return median

for parts in [4, 16, 64]:
    spark.conf.set("spark.sql.shuffle.partitions", str(parts))
    timeit(
        f"groupBy with shuffle.partitions={parts}",
        lambda: bench_df.groupBy("region").agg(F.sum("amount")).collect(),
        repeat=2,
    )

spark.conf.set("spark.sql.shuffle.partitions", "8")

## 3 · Broadcast-join thresholds

`spark.sql.autoBroadcastJoinThreshold` (default 10,485,760 = 10 MB) decides when the
small side of a join is copied to every executor instead of being shuffled.

**Caution:** If the optimizer broadcasts a table that is too large, every executor builds the hash
table in memory and the job dies with `OutOfMemoryError` — often after minutes of work.

**Note:** Pin the strategy explicitly with `F.broadcast` when you know the answer. The decision
can change between runs as statistics change.

In [ ]:
dim = (
    spark.range(500)
    .withColumn("customer_id", F.col("id").cast("int"))
    .withColumn("segment", F.lit("standard"))
)
dim.cache().count()

print("autoBroadcastJoinThreshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Broadcast vs plain join — compare the plans.
print("\n== plain join plan ==")
bench_df.join(dim, "customer_id", "inner").groupBy("segment").agg(F.sum("amount")) \
       .explain(True)

print("\n== explicit broadcast plan ==")
bench_df.join(F.broadcast(dim), "customer_id", "inner").groupBy("segment").agg(F.sum("amount")) \
       .explain(True)

## 4 · Skew detection and salt handling

**Skew** is when one key owns a disproportionate share of the rows. It produces one very
slow task while every other task finishes in seconds.

Detection: compare the maximum partition size to the median in the Spark UI's Stages
tab, or compute the ratio in SQL:

```python
key_counts.groupBy("key").count().agg(
    F.max("count").alias("max_count"),
    F.expr("percentile_approx(count, 0.5)").alias("median_count"),
)
```

**Salting** spreads a hot key across N partitions. You add a random suffix to the key,
aggregate, then aggregate the salted results again.

In [ ]:
# Create a deliberately skewed key distribution.
skewed = (
    spark.range(200_000)
    .withColumn("key", F.when(F.col("id") < 180_000, F.lit("hot")).otherwise(F.col("id") % 1000))
    .withColumn("value", F.lit(1.0))
)

print("== skew profile ==")
skewed.groupBy("key").count().agg(
    F.count("*").alias("distinct_keys"),
    F.max("count").alias("max_count"),
    F.expr("percentile_approx(count, 0.5)").alias("median_count"),
).show()

# Salting: spread the hot key across SALT_BUCKETS partitions.
SALT_BUCKETS = 32
salted = (
    skewed
    .withColumn("salt", (F.rand(42) * SALT_BUCKETS).cast("int"))
    .withColumn("salted_key", F.concat(F.col("key"), F.lit("-"), F.col("salt")))
)

print("\n== salted: group by the salted key, then re-aggregate ==")
salt_result = (
    salted
    .groupBy("key", "salted_key")
    .agg(F.sum("value").alias("partial"))
    .groupBy("key")
    .agg(F.sum("partial").alias("total"))
)
salt_result.orderBy(F.desc("total")).show(5)

# The result must equal the unsalted answer.
expected = skewed.groupBy("key").agg(F.sum("value").alias("total"))
mismatch = (
    expected.alias("e")
    .join(salt_result.alias("s"), "key")
    .filter(F.col("e.total") != F.col("s.total"))
    .count()
)
print("\nrows where salted != unsalted:", mismatch, "(0 means the salt is correct)")

## 5 · AQE — Adaptive Query Execution

AQE is **on by default in Spark 3.5**. It uses the *actual* shuffle sizes observed at
runtime to make decisions that the static optimizer could not.

| Config | Default | What it does |
|---|---|---|
| `spark.sql.adaptive.enabled` | `true` | Master switch |
| `spark.sql.adaptive.coalescePartitions.enabled` | `true` | Merges small shuffle partitions into fewer tasks |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | `64MB` | Target size for coalesced partitions |
| `spark.sql.adaptive.skewJoin.enabled` | `true` | Splits a skewed shuffle partition during a join |
| `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` | `256MB` | Size above which a partition is "skewed" |
| `spark.sql.adaptive.localShuffleReader.enabled` | `true` | Lets AQE coalesce after the shuffle is written |
| `spark.sql.adaptive.autoBroadcastJoinThreshold` | inherits the static threshold | AQE can convert a sort-merge join to a broadcast hash join at runtime |

**Note:** AQE's `coalescePartitions` is what makes the static `spark.sql.shuffle.partitions`
value a *maximum* rather than a fixed count — the final plan often has far fewer
partitions than you configured.

In [ ]:
print("== AQE settings ==")
aqe_keys = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.localShuffleReader.enabled",
    "spark.sql.adaptive.autoBroadcastJoinThreshold",
]
for key in aqe_keys:
    try:
        print(f"  {key:60s} = {spark.conf.get(key)}")
    except Exception:
        print(f"  {key:60s} = <not set>")

# AQE on vs off — the plan differs.
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("\n== plan with AQE enabled ==")
bench_df.groupBy("region").agg(F.sum("amount")).explain(True)

spark.conf.set("spark.sql.adaptive.enabled", "false")
print("\n== plan with AQE disabled ==")
bench_df.groupBy("region").agg(F.sum("amount")).explain(True)

spark.conf.set("spark.sql.adaptive.enabled", "true")

In [ ]:
# AQE coalesces small shuffle partitions at runtime.
spark.conf.set("spark.sql.shuffle.partitions", "200")   # a large static value
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print("static shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("-> AQE will typically coalesce these down to a handful of tasks at runtime.")
print("   Watch the Stages tab in the Spark UI to see the actual task count.")

spark.conf.set("spark.sql.shuffle.partitions", "8")

## 6 · Benchmark: native vs `udf` vs `pandas_udf`

This is a **real benchmark**, run in the notebook, with a warm-up and a median of three
timings.

**Caution:** **The action must consume the column the UDF produces.** `count()` does **not**
evaluate a UDF column — Spark can compute the row count from metadata and skip the UDF
entirely, which makes every implementation look identical. Use an aggregating action
such as `agg(F.sum("v"))`.

Methodology rules:

1. Discard the first run (JIT, Arrow init, metadata loading).
2. Measure with an **action**, not a plan.
3. Use `time.perf_counter()` (monotonic).
4. Test at realistic scale — a 1,000-row benchmark makes the Python UDF look *faster*.

In [ ]:
import pandas as pd

from pyspark.sql.functions import pandas_udf, udf, PandasUDFType
from pyspark.sql.types import DoubleType, StringType

# Arrow is REQUIRED for pandas_udf.
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# --- 1. Native expression (Catalyst can optimise this) -------------------
def native_factor(region):
    return {"EMEA": 1.0, "APAC": 1.2, "AMER": 0.95}.get(region, 1.0)

native_expr = F.col("amount") * F.when(F.col("region") == "APAC", F.lit(1.2)) \
                             .when(F.col("region") == "AMER", F.lit(0.95)) \
                             .otherwise(F.lit(1.0))

# --- 2. Plain Python UDF (one interpreter hop per row) -------------------
@udf(returnType=DoubleType())
def python_udf_amount(amount, region):
    factor = {"EMEA": 1.0, "APAC": 1.2, "AMER": 0.95}.get(region, 1.0)
    return round(amount * factor, 2)

# --- 3. pandas_udf (vectorised, Arrow batch transfer) --------------------
@pandas_udf(returnType=DoubleType(), functionType=PandasUDFType.SCALAR)
def pandas_udf_amount(amounts: pd.Series, regions: pd.Series) -> pd.Series:
    factors = regions.map({"EMEA": 1.0, "APAC": 1.2, "AMER": 0.95}).fillna(1.0)
    return (amounts * factors).round(2)

print("Arrow enabled:", spark.conf.get("spark.sql.execution.arrow.pyspark.enabled"))
print("all three implementations are ready.")

In [ ]:
def bench(label, expr, repeat=3, df=None):
    """Benchmark with an AGGREGATING action so the UDF column is actually evaluated."""
    timings = []
    for _ in range(repeat + 1):          # +1 warm-up run, discarded below
        t0 = time.perf_counter()
        df.withColumn("v", expr).agg(F.sum("v")).collect()
        elapsed = time.perf_counter() - t0
        timings.append(elapsed)
    timings = timings[1:]                # discard the warm-up
    timings.sort()
    median = timings[len(timings) // 2]
    print(f"{label:28s} median={median:7.3f}s   runs={[round(t, 3) for t in timings]}")
    return median

print("== REAL benchmark (1,000,000 rows) ==")
t_native = bench("native expression", native_expr, df=bench_df)
t_pyudf = bench("plain python udf", python_udf_amount(F.col("amount"), F.col("region")),
                df=bench_df)
t_pandas = bench("pandas_udf", pandas_udf_amount(F.col("amount"), F.col("region")),
                 df=bench_df)

print()
# Report the SPEEDUP each implementation represents: bigger = better.
print(f"pandas_udf vs plain python udf : {t_pyudf / t_pandas:.2f}x faster")
print(f"native expression vs plain py udf: {t_native / t_pyudf:.2f}x faster")
print(f"native expression vs pandas_udf  : {t_pandas / t_native:.2f}x faster")
print()
print("Reading the numbers:")
print(f"  pandas_udf  is {t_pyudf / t_pandas:.2f}x faster than a plain Python UDF")
print(f"  native SQL  is {t_pandas / t_native:.2f}x faster than pandas_udf")
print("  -> Arrow batching is what makes pandas_udf worthwhile, but a native")
print("     expression is still the fastest option when one exists.")

In [ ]:
# The MISLEADING benchmark, for contrast.
def bench_count_only(label, expr):
    timings = []
    for _ in range(3):
        t0 = time.perf_counter()
        bench_df.withColumn("v", expr).count()
        timings.append(time.perf_counter() - t0)
    timings.sort()
    print(f"{label:28s} median={timings[1]:7.3f}s   (count() does NOT evaluate the UDF!)")

print("== MISLEADING benchmark using count() ==")
bench_count_only("native expression", native_expr)
bench_count_only("plain python udf", python_udf_amount(F.col("amount"), F.col("region")))
bench_count_only("pandas_udf", pandas_udf_amount(F.col("amount"), F.col("region")))
print()
print("The three look almost identical -> count() never evaluates the UDF column.")
print("Always benchmark with an action that CONSUMES the column under test.")

In [ ]:
# Compare the plans: the native plan has no PythonUDF node; the UDF plans do.
print("== native plan ==")
bench_df.withColumn("v", native_expr).explain(mode="formatted")
print("\n== python udf plan ==")
bench_df.withColumn("v", python_udf_amount(F.col("amount"), F.col("region"))).explain(mode="formatted")

## 7 · Memory tuning

| Config | Meaning |
|---|---|
| `spark.executor.memory` | Executor JVM heap. **Static** — set at submit time. |
| `spark.executor.memoryOverhead` | Off-heap memory (Python workers, shuffle buffers). Default `max(384m, 0.1 × executor.memory)`. |
| `spark.executor.cores` | Task threads per executor. More cores = more concurrent tasks = more memory pressure. Usually 4–5. |
| `spark.driver.memory` | Driver JVM heap — the size of the biggest result you can pull. **Static.** |
| `spark.driver.maxResultSize` | Hard cap on a single collected result (default `1g`). |
| `spark.memory.fraction` | Share of executor heap reserved for execution vs. storage (default `0.6`). |
| `spark.sql.files.maxPartitionBytes` | Largest chunk read per partition (default `128m`). |

**Caution:** `spark.executor.memory` and `spark.driver.memory` **cannot be changed at runtime** —
the JVM heap is fixed at startup. `spark.conf.set` on them raises an exception.

**Note:** On Dataproc, `spark.executor.memory` is derived from the machine type, not set
directly.

In [ ]:
# spark.conf.get(key) RAISES when the key is absent (it is not a dict).
# The two-argument form returns a default instead.
#
# CAREFUL: for BYTE-SIZE keys the default is itself parsed as a byte string
# ("512m", "1g"), so a default like "<not set>" raises a NumberFormatException.
# Give byte-size keys a numeric default; give the others a text default.
BYTE_SIZE_KEYS = {
    "spark.executor.memory",
    "spark.executor.memoryOverhead",
    "spark.driver.memory",
    "spark.driver.maxResultSize",
    "spark.sql.files.maxPartitionBytes",
}

for key in ["spark.executor.memory", "spark.executor.memoryOverhead",
            "spark.executor.cores", "spark.executor.instances",
            "spark.driver.memory", "spark.driver.maxResultSize",
            "spark.memory.fraction", "spark.sql.files.maxPartitionBytes"]:
    default = "512m" if key in BYTE_SIZE_KEYS else "<not set on a local run>"
    print(f"{key:42s} = {spark.conf.get(key, default)}")

try:
    spark.conf.set("spark.executor.memory", "8g")
except Exception as exc:
    print("\nexecutor.memory is static ->", type(exc).__name__)

print("\nTotal cluster memory (local mode is one executor):")
print("  executor.memory + overhead =",
      spark.conf.get("spark.executor.memory", "512m"), "+",
      spark.conf.get("spark.executor.memoryOverhead", "512m"))

## 8 · Storage levels and `cache()` before fan-out

| PySpark constant | Memory | Disk | Survives executor loss |
|---|---|---|---|
| `MEMORY_ONLY` | yes (serialized) | no | no |
| `MEMORY_AND_DISK` | yes | yes | no (**the default**) |
| `MEMORY_AND_DISK_DESER` | yes (deserialized) | yes | no |
| `DISK_ONLY` | no | yes | no |
| `DISK_ONLY_2` | no | yes (2 copies) | yes |
| `MEMORY_AND_DISK_2` | yes | yes (2 copies) | yes |

**Note:** `cache()` == `persist(StorageLevel.MEMORY_AND_DISK)`. **Caching is lazy** — follow it
with an action to materialise it.

**Note:** `cache()` before a fan-out so the pipeline is computed once, not once per sink.

In [ ]:
from pyspark import StorageLevel

# A DataFrame exposes `.storageLevel` and `.is_cached` (an RDD has
# `.getStorageLevel()` instead).
cached_bench = bench_df.cache()
print("cache() storage level:", cached_bench.storageLevel)
print("is_cached           :", cached_bench.is_cached)
cached_bench.unpersist()
print("after unpersist     :", cached_bench.storageLevel, "| is_cached:", cached_bench.is_cached)

# The fan-out pattern.
fanout = (
    bench_df
    .filter(F.col("amount") > 100)
    .withColumn("amount_x2", F.col("amount") * 2)
    .cache()
)
fanout.count()      # materialise once

t0 = time.perf_counter()
fanout.write.mode("overwrite").parquet("/tmp/dataproc_notebook_09/sink_a")
fanout.write.mode("overwrite").json("/tmp/dataproc_notebook_09/sink_b")
fanout.write.mode("overwrite").csv("/tmp/dataproc_notebook_09/sink_c")
print(f"three sinks from one cached DataFrame: {time.perf_counter() - t0:.3f}s")

fanout.unpersist()


## 9 · Common anti-patterns

| Anti-pattern | Why it hurts | Fix |
|---|---|---|
| `df.rdd.map(...)` | Loses the schema, defeats Catalyst, per-row Python overhead | Use `withColumn` / `select` with native expressions |
| Plain Python `udf` when SQL will do | Opaque to Catalyst; ~5–20× slower | `F.expr` or the built-in `functions` library |
| Benchmarking with `count()` | Does not evaluate the UDF column — all results look identical | Use an aggregating action (`agg(F.sum("v"))`) |
| `collect()` on a large result | Pulls everything to one driver; 2–3× the result size in memory | `toLocalIterator()`, `write`, or aggregate first |
| `toPandas()` without `limit()` | Same problem — a full pull into the driver | `df.limit(n).toPandas()` |
| `groupByKey` instead of `reduceByKey` | Shuffles all values of a group before combining | `reduceByKey` (map-side combine) |
| `collect_set` / `collect_list` on high cardinality | Accumulates a whole group in one array | Aggregate on the element itself |
| `pivot()` on a high-cardinality column | The schema itself becomes the memory bomb | Pivot over a known small domain, or melt |
| `dropDuplicates(subset=)` for "keep latest" | Keeps an **arbitrary** row | `row_number()` with a deterministic tiebreaker |
| `union` on differently ordered schemas | Silently mis-maps columns | `unionByName` |
| Writing without `cache()` before a fan-out | Recomputes the pipeline once per sink | `cache()` + an action first |
| Hardcoding `master="local[*]"` in a submitted job | Runs the whole job on the driver node | Omit `master`; use the cluster's `yarn` |
| `df.count()` as a quality check | Counts rows, not valid rows | Count the filtered subset, and assert |
| Unbounded window over a huge dataset | A global sort plus a shuffle | `partitionBy` a high-cardinality key |
| Tiny files in GCS | Millions of objects with millisecond latency each | `repartition` / `coalesce` before writing |

## 10 · A tuning checklist

- [ ] Read `explain(True)` and count the `Exchange` nodes.
- [ ] Check `PushedFilters` and `ReadSchema` under the scan.
- [ ] Set `spark.sql.shuffle.partitions` for the actual data volume.
- [ ] Enable AQE (default on) and keep `coalescePartitions` enabled.
- [ ] Broadcast only genuinely small dimensions, and pin the strategy with `F.broadcast`.
- [ ] Profile for skew; salt the hot keys.
- [ ] Replace every Python UDF that can be written in SQL.
- [ ] `cache()` before a fan-out; `unpersist()` when done.
- [ ] Never `collect()` or `toPandas()` a large result.
- [ ] Benchmark with an aggregating action, a warm-up, and `time.perf_counter()`.
- [ ] Set `spark.executor.memory` / `spark.driver.memory` at submit time, not at runtime.

## 🔑 Key Takeaways

- `explain(True)` is free — count `Exchange` nodes and check `PushedFilters`.
- `spark.sql.shuffle.partitions` should track the data volume, not the cluster size; keep it near 1–3× the executor cores.
- AQE (on by default) coalesces small shuffle partitions and can convert joins to broadcast at runtime.
- Skew detection compares max partition size to the median; salting fixes it with a two-stage aggregation.
- **`count()` does NOT evaluate a UDF column** — benchmark with `agg(F.sum("v"))` or the numbers are meaningless.
- Measured on 1M rows: `pandas_udf` is roughly 3–4× faster than a plain Python `udf`, and a native expression beats both.
- A Python UDF is a barrier in the physical plan — Catalyst cannot push filters or prune columns past it.
- `spark.executor.memory` and `spark.driver.memory` are **static** — set them at submit time.
- `cache()` before a fan-out so the pipeline runs once; `unpersist()` when finished.
- Watch for `df.rdd.map`, unbounded windows, `collect_set` on high cardinality, and `union` on reordered schemas.

---

## 📚 Next steps

- Re-run the benchmark with 5,000,000 rows and record the medians and the input bytes read from the Spark UI.
- Continue to `10-end-to-end-pipeline.ipynb` for the capstone medallion pipeline.
